In [ ]:
import json
import requests
import torch
# URL of the VG-SGG dicts file
url = "https://svl.stanford.edu/projects/scene-graph/dataset/VG-SGG-dicts.json"

# Download the file
response = requests.get(url)
response.raise_for_status()  # raise error if download failed

# Parse JSON
vg_dicts = response.json()

# The object categories
object_classes = vg_dicts["idx_to_label"] 
predicates =list(vg_dicts["idx_to_predicate"].values())
subjects = objects = list(object_classes.values())


In [ ]:
import torch
from transformers import BertTokenizer, BertModel,BertForMaskedLM, AutoConfig
import pickle
from tqdm import tqdm

# ----------------------------
# Config
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "bert-base-uncased"
alpha = 0.6   # weight for [MASK] (relation-focused)
batch_size = 16


templates = [
    "[CLS] {subj} is [MASK] the {obj}. [SEP]",
    "[CLS] The {subj} is [MASK] the {obj} . [SEP]",
    "[CLS] {subj} [MASK] with the {obj}. [SEP]",
]

# ----------------------------
# Load BERT
# ----------------------------
config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name,config=config).to(device)
model.eval()
predicate_ids = tokenizer.convert_tokens_to_ids(predicates)

# ----------------------------
# Encode function
# ----------------------------
        
def encode_batch(batch_texts,predicate_ids):
    enc = tokenizer(batch_texts, return_tensors="pt",
                    padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**enc)
        hidden = outputs.hidden_states[-1]  # [B, L, 768]
        logits = outputs.logits
    # Find [MASK] index for each sequence
    mask_indices = (enc["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)
    # Each row in mask_indices: (batch_idx, mask_pos)
    mask_embs = torch.zeros((3,768))
    score = 0
    print(mask_indices)
    for b, pos in mask_indices:
        mask_embs[b] = hidden[b, pos, :]
        mask_logits = logits[b,pos,predicate_ids]
        print(len(mask_logits))
        score += torch.sum(mask_logits)
    score = score / 3
    return mask_embs.cpu(),score  # [B,768]

# ----------------------------
# Build embeddings tensor
# ----------------------------
S, O, T = len(subjects), len(objects), len(templates)
all_embs = torch.zeros((S + 1, O + 1, T, 768))
all_scores = torch.zeros((S + 1, O + 1))
for si, subj in enumerate(subjects,start=1):
    for oi, obj in enumerate(objects,start=1):
        batch_texts = [temp.format(subj=subj, obj=obj) for temp in templates]
        
        # Process in small batch (T usually small anyway)
        embs,score = encode_batch(batch_texts,predicate_ids)  # [T,768]
        all_scores[si, oi] = score
        all_embs[si, oi, :, :] = embs

# ----------------------------
# Save to pickle
# ----------------------------

with open("so_template_embeddings.pkl", "wb") as f:
    pickle.dump(all_embs, f)
with open("scores.pkl", "wb") as f:
    pickle.dump(all_scores, f)
print("Saved embeddings:", all_embs.shape,all_scores.shape)
